# M1 — Literature Survey

**Project:** Attacking and Defending AI Résumé Screeners against Indirect Prompt Injection  
**Milestone:** M1 — Literature Survey  
**Date:** 26 October 2026

> **Purpose.** This note explains why the project studies hidden instructions in résumés, compares five relevant papers, and justifies the paper used as the M2 reproduction baseline. It is written to establish the research direction before implementation.


## Start here: project overview

AI screening systems may read résumé text and use it to rank or select candidates. An applicant could place instructions inside a résumé that are visible to the text extractor but difficult for a person to notice. This project studies that risk and ways to reduce it.

**Research sequence:** M1 surveys the literature and selects an anchor paper. M2 reconstructs that paper’s benchmark and evaluates its main defense claims. A later M3 milestone is planned to test defenses with a real language model and realistic PDF résumés.

**Main conclusion from the literature review:** ResumeShield is the best fit for the M2 baseline because it focuses on résumé screening and provides a small benchmark, measurable outcomes, and a defense that can be implemented with limited compute. Its simulated screener is an important limitation; reproducing its results does not by itself show how a real hiring system or LLM would behave.

### Terms used in this note

- **LLM (large language model):** a model that processes and generates text.
- **Prompt injection:** text that tries to make a model follow instructions that conflict with its intended task.
- **Indirect prompt injection:** instructions placed in content the system reads, such as a résumé, rather than entered directly by the user.
- **Concealed or hidden text:** text that a parser or model may receive but a human reviewer may not readily see.
- **Attack success rate (ASR):** the fraction of injected cases in which the attack changes the screening outcome as defined by the experiment.
- **Channel separation:** treating document content as untrusted data, separate from the instructions that govern the model.

### How to read the sections

Sections 1–2 define the research area and question. Sections 3–4 summarize and compare five papers. Sections 5–7 explain the anchor-paper choice and assess reproducibility. Section 8 lists the sources.


## 1. Research Area

**Area: Indirect prompt injection — attacks, measurement, and defenses — in LLM-based document screening, with a focus on AI resume/CV screening.**

Large language models (LLMs) increasingly operate inside applications that process untrusted third-party content, including web pages, email, PDFs, and—within this project’s scope—**applicant résumés**. Because instructions and data are represented in the same text stream, a document that appears to contain ordinary candidate information can also carry instructions intended for the model. This is known as **indirect prompt injection** (OWASP LLM01:2025). In résumé screening, an attacker may conceal a payload from human reviewers using white-on-white text, zero-size text, off-page positioning, document metadata, zero-width characters, or homoglyphs.

This area supports a reproducible, extensible project for four reasons: the threat has been measured in production corpora; document-based attacks are inexpensive to reproduce; candidate defenses based on text visibility and classification do not require large-model training; and prior work provides an open benchmark alongside a stated extension opportunity.

**Scope.** This survey focuses on indirect prompt injection carried inside documents that a screening system ingests. It does not cover standalone chatbot jailbreaks or direct prompt injection entered by a user in the chat interface.

---

## 2. Initial Research Problem (stated before anchor selection)

> **AI resume screeners that read resume text are vulnerable to indirect prompt injection via hidden text. How large is the attack success rate against a realistic screener, and can a lightweight, content‑agnostic defense reduce that rate substantially while preserving legitimate screening behavior?**

Sub‑questions that guided the literature search:
1. Is the threat demonstrated only conceptually, or does it occur in the wild and at what rate?
2. What concealment techniques are used, and which defenses have been proposed (filtering, detection, structural/channel separation, encoding)?
3. Is there a reusable **benchmark** and **evaluation metric** (e.g., attack success rate) to measure "before vs. after" a defense?
4. Which prior work is reproducible on free compute and leaves a clear, testable extension?

§5 refines this into a single concrete research question once the anchor is fixed.

---

## 3. Five Selected Papers

The five papers below represent foundational work, real-world measurement, document-level detection, and general evaluation methods. The comparison appears in §4; additional related work is discussed in §6.

**[P1] Greshake, Abdelnabi, Mishra, Endres, Holz, Fritz (2023).** *Not what you've signed up for: Compromising Real‑World LLM‑Integrated Applications with Indirect Prompt Injection.* arXiv:2302.12173 (AISec '23 workshop). — The foundational paper that names and demonstrates **indirect** prompt injection. It argues LLM‑integrated apps "blur the line between data and instructions," builds a security taxonomy (data theft, worming, remote control, ecosystem contamination), and shows practically viable attacks on Bing Chat (GPT‑4), code‑completion engines, and synthetic GPT‑4 apps. It establishes *why* the resume‑screening threat exists but gives no quantitative benchmark or defense. Code (proof‑of‑concept demos): https://github.com/greshake/llm-security.

**[P2] Zhang, Jia, Tan, Jiang, Gong, Chen, Song (2026).** *Measuring Real‑World Prompt Injection Attacks in LLM‑based Resume Screening.* arXiv:2605.28999 — USENIX Security 2026. — The first **systematic measurement** of whether this attack actually happens in practice. The authors build resume‑tailored injection detectors, validate them manually, and run them over **~200K real resumes** gathered over several years by hireEZ. Headline results: roughly **1% of resumes contain hidden injections**, prevalence has risen noticeably in the past 1–2 years, and **>90% of injected prompts avoid explicit instructions** (they inject fake skills/experience — "data injection" — rather than overt commands). Code: https://github.com/UNITES-Lab/resume-injection-measurement (the 200K corpus is proprietary and not released).

**[P3] Barach (2026).** *ResumeShield: Channel Separation and an Open Benchmark for Indirect Prompt Injection in AI Resume Screening.* arXiv:2609.20188. — **← ANCHOR PAPER.** Provides both an **open, reproducible benchmark** (104 seeded synthetic documents spanning 9 concealment techniques × 2 payload families → 72 injected cases) and a **defense**: a three‑stage filtering stack (e.g., markup stripping, instruction redaction) plus a fourth architectural stage, **channel separation**, which fences resume content inside an explicitly inert "data" channel that the trusted system prompt declares non‑executable. On a *deterministic simulated screener*, the naive pipeline's attack success rate is **1.000 (72/72)** and the defended pipeline's is **0.000**; hidden‑prompt detection reaches **precision 1.000, recall 0.944, F1 0.971, FPR 0.000**. Released under Apache 2.0, Python‑stdlib‑only, ~0.5 ms/document on one CPU core. The authors explicitly list "replace the simulated backend with hosted [real LLM] models" as future work — the gap this project fills.

**[P4] Murray (2025).** *PhantomLint: Principled Detection of Hidden LLM Prompts in Structured Documents.* arXiv:2508.17884. — A **detection‑side** paper: the first "principled" approach to finding hidden LLM prompts in structured documents (PDF + HTML), motivated by résumé screening and peer review. Evaluated on a corpus of **3,402 documents** (preprints, CVs, theses), it reports a very low **false‑positive rate ≈ 0.092%** and claims general applicability across many hiding methods. It validates the core idea behind this project's first defense check — comparing what is extractable/hidden against what is visibly rendered — but reports no precision/recall in the abstract and surfaces no public code.

**[P5] Liu, Jia, Jia, Gong (2024).** *Formalizing and Benchmarking Prompt Injection Attacks and Defenses.* arXiv:2310.12815 — USENIX Security 2024. — The **standard evaluation framework** for the field. It formalizes prompt injection (existing attacks become special cases), proposes a strong *combined* attack, and systematically benchmarks **5 attacks × 10 defenses × 10 LLMs × 7 NLP tasks**, with well‑defined metrics (ASV/ASR/PNA, etc.). The tooling is mature and public — https://github.com/liu00222/Open-Prompt-Injection (MIT, ~500★, includes a notebook). It is general‑domain (sentiment, spam) rather than resume‑specific, which makes it the ideal **measurement scaffold** to adapt rather than the domain anchor.

---

## 4. Comparison Table

The table compares the five papers across nine dimensions. P3 is the selected anchor paper. **PI** means prompt injection; **ASR** means attack success rate. Values are summarized from the cited papers and should be interpreted in light of each study’s methods and dataset.

| Dimension | **[P1] Greshake 2023** | **[P2] Zhang 2026 (USENIX'26)** | **[P3] ResumeShield / Barach 2026 ★ANCHOR** | **[P4] PhantomLint / Murray 2025** | **[P5] Liu 2024 (USENIX'24)** |
|---|---|---|---|---|---|
| **Research problem** | Can attackers compromise LLM‑integrated apps by planting instructions in *retrieved* data? | Does PI actually occur in real resume screening, and how prevalent/impactful? | How to defend AI resume screeners against indirect PI, with an open benchmark to measure it. | Detect hidden LLM prompts embedded in structured documents before they reach the model. | Formalize PI and systematically benchmark attacks vs. defenses across models/tasks. |
| **Method** | Threat model + security taxonomy; inject prompts into content the app ingests; demonstrate on real & synthetic apps. | Build resume‑tailored PI detectors; manual validation on a small set; measurement sweep over full corpus. | 3‑stage content filtering (markup strip, instruction redaction, …) **+ channel separation** (inert fenced data channel). | "Principled" structural detection of hidden prompts; prototype tool across many hiding methods. | Formal framework; new *combined* attack; unified attack/defense evaluation harness. |
| **Dataset** | No benchmark; qualitative demos. | ~200K **real** resumes (hireEZ, multi‑year). **Not public.** | **104** synthetic docs; 9 concealment techniques × 2 payload families → 72 injected cases. **Public (synthetic).** | **3,402** docs (PDF+HTML): preprints, CVs, theses. Availability not stated. | **7 NLP tasks** (e.g., SST‑2 sentiment, SMS spam). Public via repo. |
| **Model** | GPT‑4 (Bing Chat), GPT‑3/LangChain, code‑completion engines. | Screening LLM(s) not stated; detectors are custom. | **Deterministic *simulated* screener** (real LLMs = future work). | Prototype tool (structural); LLM not required for detection. | **10 LLMs** (PaLM 2, Llama family, OpenAI GPT, …). |
| **Metrics** | Qualitative ("practically viable"); no ASR. | Detector precision (manually validated); prevalence rate; share of non‑explicit injections. | **ASR** (outcome‑change), detection **precision/recall/F1/FPR**, ablation. | **False‑positive rate** (≈0.092%); precision/recall not in abstract. | **ASV/ASR/PNA** and related; utility under no attack. |
| **Code / data availability** | Code public (PoC demos); no dataset. | Code public; **dataset proprietary**. | **Apache 2.0, stdlib‑only; synthetic benchmark released** (verify exact repo URL on paper title page). | **No public code found**; corpus availability not stated. | **Code public & mature** (MIT, notebook); checkpoints downloadable. |
| **Main finding** | Indirect PI is practically viable; retrieved prompts can act like arbitrary code execution; mitigations lacking. | ~**1%** of real resumes are injected; prevalence rising; **>90% non‑explicit** (data injection). | Naive ASR **1.000 → 0.000** defended; channel‑separation alone removes all attacks; filters‑without‑fence leave **16.7%**; "**concealment dilemma**": evading detection forces the payload to be visible. | Detects hidden prompts across many methods with very low FPR (**~0.092%**). | Combined attack is strong; **most existing defenses have limited robustness** across 10 LLMs / 7 tasks. |
| **Limitation** | Demonstrative/qualitative; no defense evaluated. | Dataset not reproducible; measures prevalence, not end‑to‑end ASR on a live screener. | **Simulated** (not real‑LLM) screener; synthetic corpus shorter/more uniform than real resumes; v1 handles only text‑extractable formats (**not PDF yet**); risk weights uncalibrated. | Detection only (no end‑to‑end screener); no precision/recall reported; no public code. | General‑domain tasks; no document‑rendering / hidden‑text (visibility) attacks. |
| **Possible extension** | Quantify ASR; evaluate defenses; specialize to resume screening. | Measure real ranking impact on live LLM screeners; test defenses at scale. | **Run the defense against real LLMs on realistic/PDF resumes; add a complementary visibility‑diff + instruction‑classifier defense** (this project). | Wire the detector into a screening pipeline; measure end‑to‑end ASR drop; release code. | Add a **resume‑screening task + concealment attacks** to the harness; reuse as this project's metric engine. |

---

## 5. Selected Anchor Paper & Justification

### Anchor: **[P3] ResumeShield — Channel Separation and an Open Benchmark for Indirect Prompt Injection in AI Resume Screening** (Barach, 2026; arXiv:2609.20188)

**Rationale.** ResumeShield is the closest match to the project’s topic: indirect prompt injection in AI résumé screening. It also provides a benchmark, a defense, and a stated extension opportunity. Greshake et al. establish the threat but do not provide a benchmark; Zhang et al. measure the threat using a non-public corpus; PhantomLint focuses on detection; and Liu et al. provide general-purpose tooling rather than a résumé-specific study. ResumeShield therefore best meets the project’s selection criteria:

**1. Research relevance.** The study addresses a security problem measured in real résumé corpora (P2 reports approximately 1% prevalence). It contributes an open benchmark, a channel-separation defense, and the **concealment dilemma**: a payload that evades a detector by remaining visible may also become apparent during human review. This claim gives the project a concrete result to test.

**2. Reproducibility.** The benchmark is synthetic and, according to the paper, released under Apache 2.0. Its reported structure—nine concealment techniques, two payload families, and 104 documents—is documented, as are the metrics. The deterministic backend supports repeatable results. Reproducibility is nevertheless partial: evaluation uses a simulated screener rather than a live LLM, and the code/data URL still requires confirmation (see §7).

**3. Computationally feasible (free compute).** The defense is **Python standard library only**, runs on **one CPU core at ~0.5 ms/document (~2,066 docs/s)**, needs **no GPU and no model training**, and the corpus is tiny (104 docs). It runs end‑to‑end on Google Colab / Kaggle free tiers in seconds. Adding a real LLM screener on top fits comfortably within a free API tier (e.g., `gpt-4o-mini`) or a small open model.

**4. Extensibility.** The paper identifies evaluation with hosted models as future work and notes limitations around document formats and fabricated claims in visible text. This project proposes a complementary defense: compare parser-extracted PDF text with visibly rendered text, then use an instruction-intent classifier to redact flagged spans before model ingestion.

### Refined Research Question (the project's thesis)

> **ResumeShield drives attack success to zero with channel separation — but only against a deterministic simulated screener on short synthetic documents. Does a visibility‑based defense — comparing PDF‑extracted text against visibly‑rendered text to surface hidden content, combined with an instruction‑intent classifier that redacts flagged spans before ingestion — reduce the attack success rate of hidden‑text injections against a *real* LLM resume screener (e.g., `gpt‑4o‑mini` / Llama‑3) on realistic PDF resumes, and how does it compare to ResumeShield's channel separation?**

**Testable hypothesis / success target.** Replicate a high baseline on an *undefended real‑LLM* screener (target ASR ≈ 70%+ across the concealment techniques), then show the visibility‑diff + classifier defense cuts ASR to **< 5%** while leaving clean‑resume rankings unchanged (false‑positive/utility check) — mirroring ResumeShield's 1.000 → 0.000 on a simulated backend, but now on real models and real PDFs. ResumeShield's filtering/metric design and the "concealment dilemma" give a principled baseline to compare against; P5's harness (`Open-Prompt-Injection`) supplies the ASR measurement scaffold.

---

## 6. Additional Related Work (breadth beyond the five)

The following work provides additional context for the anchor paper and informs the proposed defense design:

- **Hines et al. (2024), Microsoft — *Defending Against Indirect Prompt Injection Attacks with Spotlighting* (arXiv:2403.14720).** Three defenses that mark untrusted content so the model can tell data from instructions: **delimiting**, **datamarking** (a token woven through the text), and **encoding** (e.g., Base64). Encoding reached near‑0% ASR on capable models (GPT‑4) but degraded weaker ones; datamarking had negligible task‑utility cost. Conceptually adjacent to ResumeShield's channel separation.
- **Chen et al. (2024), UC Berkeley — *StruQ: Defending Against Prompt Injection with Structured Queries* (arXiv:2402.06363).** Separates instructions from data via structured queries + fine‑tuning; cuts manual‑attack ASR to <2% on Llama/Mistral, though optimization‑based attacks (GCG) still partly succeed. A training‑based alternative to the project's inference‑time defense.
- **Beurer‑Kellner et al. (2025) — *Design Patterns for Securing LLM Agents against Prompt Injections* (arXiv:2506.08837).** Uses resume screening as a running threat‑model example; a useful source of defensive patterns.
- **Wallace et al. (2024), OpenAI — *The Instruction Hierarchy.*** Training models to prioritize privileged (system) over untrusted (data) instructions — the model‑level counterpart to channel separation.

---

## 7. Reproducibility Checklist (for the anchor: ResumeShield, Barach 2026)

| # | Checklist item | Answer | Evidence / notes |
|---|---|---|---|
| 1 | **Public dataset available?** | **Yes (partial)** | Synthetic benchmark of 104 documents released under **Apache 2.0**. Real resumes are *not* included (by design). Confirm the exact hosting URL from the paper's title page. |
| 2 | **Dataset size documented?** | **Yes** | 104 documents; **9 concealment techniques × 2 payload families → 72 injected cases** (+ clean documents). |
| 3 | **Public code available?** | **Yes (claimed; verify URL)** | Paper states the tool is open‑source under Apache 2.0 "at the address on the title page." Python stdlib only; CI across 4 Python versions. *Action: open the PDF title page / search GitHub for the exact repo.* |
| 4 | **Public pretrained model available?** | **N/A (none needed)** | The screener is a **deterministic simulated backend**; the defense is rule/structure‑based. No pretrained model to download. (The project will *add* an LLM screener via API or a small open model.) |
| 5 | **Hardware requirement known?** | **Yes (minimal)** | Runs on **one CPU core**, **~0.484 ms/document (~2,066 docs/s)**. **No GPU required.** |
| 6 | **Training time known?** | **Yes (effectively zero)** | No model training; filter/architecture‑based defense + simulated screener. |
| 7 | **Dependencies listed?** | **Yes** | **Python standard library only** — no external runtime dependencies. |
| 8 | **Evaluation metric clearly defined?** | **Yes** | **ASR** (counted only when the screening *outcome changes*), detection **precision / recall / F1**, **FPR**, plus an ablation over filtering stages. |
| 9 | **Feasible baseline runnable on Colab/Kaggle/free GPU?** | **Yes (easily)** | CPU‑only, stdlib, 104‑doc corpus → runs in seconds on any free tier; no GPU needed. Adding `gpt‑4o‑mini` (free/low‑cost API) or a small open model for the real‑LLM extension is also free‑tier‑feasible. |
| 10 | **Are reported results sufficiently reproducible?** | **Partially** | The deterministic backend + released synthetic benchmark make the paper's **own numbers (ASR 1.000 → 0.000; detection P/R 1.000/0.944) highly repeatable**. However, results are against a **simulated** screener on short synthetic docs, so **real‑world validity is unproven** — the authors' acknowledged limitation and the project's extension target. |

**Overall assessment:** Reproducibility is strong for the documented benchmark, metrics, compute requirements, and dependencies. Two items remain uncertain: confirmation of the code/data URL and external validity beyond the simulated screener. These define clear follow-up work for the project.

---

## 8. References

1. K. Greshake, S. Abdelnabi, S. Mishra, C. Endres, T. Holz, M. Fritz. *Not what you've signed up for: Compromising Real‑World LLM‑Integrated Applications with Indirect Prompt Injection.* 2023. arXiv:2302.12173 — https://arxiv.org/abs/2302.12173 · Code: https://github.com/greshake/llm-security
2. M. Zhang, Y. Jia, Z. Tan, S. Jiang, N. Z. Gong, T. Chen, D. Song. *Measuring Real‑World Prompt Injection Attacks in LLM‑based Resume Screening.* USENIX Security 2026. arXiv:2605.28999 — https://arxiv.org/abs/2605.28999 · Code: https://github.com/UNITES-Lab/resume-injection-measurement
3. J. Barach. *ResumeShield: Channel Separation and an Open Benchmark for Indirect Prompt Injection in AI Resume Screening.* 2026. arXiv:2609.20188 — https://arxiv.org/abs/2609.20188  **(ANCHOR)**
4. T. Murray. *PhantomLint: Principled Detection of Hidden LLM Prompts in Structured Documents.* 2025. arXiv:2508.17884 — https://arxiv.org/abs/2508.17884
5. Y. Liu, Y. Jia, R. Jia, N. Z. Gong. *Formalizing and Benchmarking Prompt Injection Attacks and Defenses.* USENIX Security 2024. arXiv:2310.12815 — https://arxiv.org/abs/2310.12815 · Code: https://github.com/liu00222/Open-Prompt-Injection

*OWASP LLM01:2025 (Prompt Injection) — OWASP Top 10 for LLM Applications.*

---

*Note on sources: arXiv identifiers, authorship, datasets, metrics, and code‑availability statements above were verified against arXiv abstract/HTML pages and project repositories in October 2026. Two items are flagged for direct confirmation before the next milestone: the exact ResumeShield code/data repository URL, and whether the PhantomLint prototype has since been released publicly.*